# 04. DB 볼륨 추정 + 종합 요약

**목표**: 3개 DB별 용량 추정 + 전체 EDA 종합 요약

**의존**: `eda_output/phase1~6 결과 전체`

**산출물**: `eda_output/phase7_projections.json`

In [1]:
# ── 환경 설정 ──────────────────────────────────────────────
import sys
from pathlib import Path

BACKEND_DIR = Path.cwd().parent.parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from scripts.eda.common import load_result, save_result
from scripts.eda.data_registry import CATEGORIES

pio.templates.default = "plotly_white"

# 전체 결과 로드
phase1 = load_result("phase1_inventory")
phase2 = load_result("phase2_schema")
phase3 = load_result("phase3_quality")
phase4 = load_result("phase4_text")
phase5 = load_result("phase5_temporal")
phase6 = load_result("phase6_relationships")

print("모든 Phase 결과 로드 완료")

모든 Phase 결과 로드 완료


## 1. PostgreSQL 용량 추정

In [2]:
# ── PostgreSQL 테이블별 용량 추정 ────────────────────────
# 카테고리별 레코드 수 합산
df_inv = pd.DataFrame(phase1)
cat_records = df_inv.groupby("category").agg(
    total_records=("record_count", "sum"),
    total_size_mb=("size_mb", "sum"),
).reset_index()

# 평균 행 크기 추정 (JSON 크기 / 레코드 수, 인덱스 오버헤드 1.5x)
PG_OVERHEAD = 1.5  # 인덱스 + 튜플 헤더 오버헤드
PG_TEXT_RATIO = 0.6  # JSON 대비 PostgreSQL TEXT 저장 비율 (압축)

pg_estimates = []
for _, row in cat_records.iterrows():
    cat_key = row["category"]
    cat_label = CATEGORIES.get(cat_key, {}).get("label", cat_key)
    records = int(row["total_records"])
    json_mb = row["total_size_mb"]

    if records == 0:
        continue

    avg_row_kb = (json_mb * 1024 * PG_TEXT_RATIO) / records
    table_mb = (records * avg_row_kb / 1024) * PG_OVERHEAD

    pg_estimates.append({
        "카테고리": cat_label,
        "category": cat_key,
        "레코드 수": records,
        "평균 행 크기 (KB)": round(avg_row_kb, 2),
        "추정 용량 (MB)": round(table_mb, 1),
    })

df_pg = pd.DataFrame(pg_estimates).sort_values("추정 용량 (MB)", ascending=True)
total_pg_mb = df_pg["추정 용량 (MB)"].sum()
print(f"PostgreSQL 추정 총 용량: {total_pg_mb:.0f} MB ({total_pg_mb/1024:.1f} GB)")

PostgreSQL 추정 총 용량: 4454 MB (4.3 GB)


In [3]:
# ── PostgreSQL 테이블별 추정 용량 stacked bar ────────────
fig = px.bar(
    df_pg,
    x="추정 용량 (MB)",
    y="카테고리",
    orientation="h",
    title=f"PostgreSQL 테이블별 추정 용량 (총 {total_pg_mb:.0f} MB)",
    hover_data=["레코드 수", "평균 행 크기 (KB)"],
    color="추정 용량 (MB)",
    color_continuous_scale="Blues",
)
fig.update_layout(height=500, showlegend=False)
fig.show()

## 2. LanceDB 용량 추정

In [4]:
# ── LanceDB 카테고리별 청크/벡터 추정 ────────────────────
VECTOR_DIM = 1024
BYTES_PER_FLOAT = 4
CHUNK_CHAR_LIMIT = 1250
OVERLAP_CHARS = 200
METADATA_BYTES_PER_ROW = 500  # 메타데이터 오버헤드

lance_estimates = []

for cat_key in CATEGORIES:
    cat_label = CATEGORIES[cat_key]["label"]
    text_fields = CATEGORIES[cat_key].get("text_fields", [])

    # 해당 카테고리 레코드 수
    cat_inv = [f for f in phase1 if f["category"] == cat_key]
    total_records = sum(f["record_count"] for f in cat_inv)

    if total_records == 0 or not text_fields:
        continue

    # 텍스트 길이로 예상 청크 수 계산
    ts = phase4.get(cat_key, {})
    if not ts or "fields" not in ts:
        avg_chunks_per_record = 2.0  # 기본 추정
    else:
        field_stats = ts["fields"]
        # 첫 번째 텍스트 필드의 평균 길이 사용
        first_field = list(field_stats.keys())[0] if field_stats else None
        if first_field and "mean" in field_stats[first_field]:
            effective_chunk = CHUNK_CHAR_LIMIT - OVERLAP_CHARS
            avg_chunks_per_record = max(1, field_stats[first_field]["mean"] / effective_chunk)
        else:
            avg_chunks_per_record = 2.0

    estimated_chunks = int(total_records * avg_chunks_per_record)

    # 벡터 저장량: chunks * dim * 4bytes
    vector_mb = (estimated_chunks * VECTOR_DIM * BYTES_PER_FLOAT) / (1024 * 1024)
    # 메타데이터
    meta_mb = (estimated_chunks * METADATA_BYTES_PER_ROW) / (1024 * 1024)
    total_mb = vector_mb + meta_mb

    lance_estimates.append({
        "카테고리": cat_label,
        "category": cat_key,
        "레코드 수": total_records,
        "예상 청크 수": estimated_chunks,
        "평균 청크/레코드": round(avg_chunks_per_record, 1),
        "벡터 (MB)": round(vector_mb, 1),
        "메타 (MB)": round(meta_mb, 1),
        "합계 (MB)": round(total_mb, 1),
    })

df_lance = pd.DataFrame(lance_estimates)
total_chunks = df_lance["예상 청크 수"].sum()
total_lance_mb = df_lance["합계 (MB)"].sum()
print(f"LanceDB 추정: {total_chunks:,} 청크, {total_lance_mb:.0f} MB ({total_lance_mb/1024:.1f} GB)")

LanceDB 추정: 845,212 청크, 3705 MB (3.6 GB)


In [5]:
# ── 카테고리별 예상 청크 수 treemap ─────────────────────
fig = px.treemap(
    df_lance,
    path=["카테고리"],
    values="예상 청크 수",
    title=f"LanceDB 카테고리별 예상 청크 수 (총 {total_chunks:,}개)",
    color="합계 (MB)",
    color_continuous_scale="Greens",
    hover_data=["레코드 수", "평균 청크/레코드", "합계 (MB)"],
)
fig.update_layout(height=500)
fig.show()

## 3. Neo4j 용량 추정

In [6]:
# ── Neo4j 노드/엣지 추정 ────────────────────────────────
# 현재 구축된 데이터 + 확장 가능 데이터
neo4j_current = {
    "nodes": [
        {"타입": "Statute", "수": 5572, "상태": "구축됨"},
        {"타입": "Case", "수": 65107, "상태": "구축됨"},
        {"타입": "Alias", "수": 69, "상태": "구축됨"},
    ],
    "edges": [
        {"타입": "HIERARCHY_OF", "수": 3624, "상태": "구축됨"},
        {"타입": "CITES", "수": 72414, "상태": "구축됨"},
        {"타입": "CITES_CASE", "수": 87654, "상태": "구축됨"},
        {"타입": "RELATED_TO", "수": 93, "상태": "구축됨"},
        {"타입": "ALIAS_OF", "수": 69, "상태": "구축됨"},
    ],
}

# 확장 예상 (헌재, 행정심판 등 추가 시)
constitutional_records = sum(
    f["record_count"] for f in phase1 if f["category"] == "constitutional"
)
admin_records = sum(
    f["record_count"] for f in phase1 if f["category"] == "administration"
)

neo4j_expanded = {
    "nodes": [
        {"타입": "Constitutional", "수": constitutional_records, "상태": "확장 가능"},
        {"타입": "AdminCase", "수": admin_records, "상태": "확장 가능"},
    ],
    "edges": [
        {"타입": "CITES (헌재→법령)", "수": int(constitutional_records * 0.4), "상태": "확장 가능"},
        {"타입": "CITES (행정→법령)", "수": int(admin_records * 0.3), "상태": "확장 가능"},
    ],
}

all_nodes = neo4j_current["nodes"] + neo4j_expanded["nodes"]
all_edges = neo4j_current["edges"] + neo4j_expanded["edges"]

total_nodes = sum(n["수"] for n in all_nodes)
total_edges = sum(e["수"] for e in all_edges)
print(f"Neo4j 추정: {total_nodes:,} 노드, {total_edges:,} 엣지")

Neo4j 추정: 136,720 노드, 186,817 엣지


In [7]:
# ── Neo4j 노드/엣지 추정 sunburst 차트 ─────────────────
sunburst_data = []
for n in all_nodes:
    sunburst_data.append({
        "항목": n["타입"],
        "유형": "노드",
        "수": n["수"],
        "상태": n["상태"],
    })
for e in all_edges:
    sunburst_data.append({
        "항목": e["타입"],
        "유형": "엣지",
        "수": e["수"],
        "상태": e["상태"],
    })

df_neo = pd.DataFrame(sunburst_data)

fig = px.sunburst(
    df_neo,
    path=["유형", "상태", "항목"],
    values="수",
    title=f"Neo4j 그래프 구조 (노드 {total_nodes:,} + 엣지 {total_edges:,})",
    color="상태",
    color_discrete_map={"구축됨": "#4472C4", "확장 가능": "#ED7D31"},
)
fig.update_layout(height=600)
fig.show()

## 4. 3개 DB 통합 용량 비교

In [8]:
# ── 3개 DB 통합 용량 비교 ────────────────────────────────
# Neo4j 용량 추정: 노드 1KB + 엣지 0.5KB 기준
neo4j_mb = (total_nodes * 1 + total_edges * 0.5) / 1024

db_comparison = pd.DataFrame([
    {"DB": "PostgreSQL", "추정 용량 (MB)": round(total_pg_mb, 0), "용도": "원본 저장 + 검색"},
    {"DB": "LanceDB", "추정 용량 (MB)": round(total_lance_mb, 0), "용도": "벡터 임베딩 + RAG"},
    {"DB": "Neo4j", "추정 용량 (MB)": round(neo4j_mb, 0), "용도": "관계 그래프"},
])

fig = px.bar(
    db_comparison,
    x="DB",
    y="추정 용량 (MB)",
    color="DB",
    title="3개 DB 통합 용량 비교",
    hover_data=["용도"],
    color_discrete_sequence=["#4472C4", "#70AD47", "#ED7D31"],
    text="추정 용량 (MB)",
)
fig.update_traces(texttemplate="%{text:.0f} MB", textposition="outside")
fig.update_layout(height=400, showlegend=False)
fig.show()

## 5. 종합 요약 대시보드

In [9]:
# ── 핵심 수치 indicator cards ────────────────────────────
total_files = len(phase1)
total_records = sum(f["record_count"] for f in phase1)
total_size_gb = sum(f["size_mb"] for f in phase1) / 1024
total_db_gb = (total_pg_mb + total_lance_mb + neo4j_mb) / 1024

fig = make_subplots(
    rows=2, cols=3,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
    ],
)

fig.add_trace(go.Indicator(
    mode="number",
    value=total_files,
    title={"text": "데이터 파일"},
    number={"suffix": "개"},
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number",
    value=total_records,
    title={"text": "총 레코드"},
    number={"valueformat": ",.0f", "suffix": "건"},
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number",
    value=total_size_gb,
    title={"text": "원본 데이터"},
    number={"valueformat": ".1f", "suffix": " GB"},
), row=1, col=3)

fig.add_trace(go.Indicator(
    mode="number",
    value=len(CATEGORIES),
    title={"text": "카테고리"},
    number={"suffix": "개"},
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode="number",
    value=total_chunks,
    title={"text": "예상 청크"},
    number={"valueformat": ",.0f", "suffix": "개"},
), row=2, col=2)

fig.add_trace(go.Indicator(
    mode="number",
    value=total_db_gb,
    title={"text": "예상 DB 총량"},
    number={"valueformat": ".1f", "suffix": " GB"},
), row=2, col=3)

fig.update_layout(
    title="EDA 종합 요약",
    height=400,
)
fig.show()

In [10]:
# ── DB 구축 우선순위 권고 테이블 ─────────────────────────
priority_data = [
    {
        "우선순위": 1,
        "카테고리": "판례 + 법령",
        "DB": "LanceDB + PostgreSQL",
        "이유": "RAG 검색의 핵심 데이터, 기존 임베딩 구축 완료",
        "레코드 수": f"{sum(f['record_count'] for f in phase1 if f['category'] in ('precedent', 'law')):,}",
    },
    {
        "우선순위": 2,
        "카테고리": "헌재 + 행정심판 + 특별행정심판",
        "DB": "LanceDB + PostgreSQL",
        "이유": "법률 검색 범위 확장, 유사한 스키마 구조",
        "레코드 수": f"{sum(f['record_count'] for f in phase1 if f['category'] in ('constitutional', 'administration', 'special_tribunal')):,}",
    },
    {
        "우선순위": 3,
        "카테고리": "위원회 + 부처 해석례",
        "DB": "LanceDB + PostgreSQL",
        "이유": "행정 해석례 검색, 다수 파일 통합",
        "레코드 수": f"{sum(f['record_count'] for f in phase1 if f['category'] in ('committee', 'cgm_expc', 'legislation')):,}",
    },
    {
        "우선순위": 4,
        "카테고리": "법률용어 + 조약 + 행정규칙",
        "DB": "PostgreSQL (검색 보조)",
        "이유": "MeCab 보강, 참고 데이터",
        "레코드 수": f"{sum(f['record_count'] for f in phase1 if f['category'] in ('law_term', 'treaty', 'school')):,}",
    },
    {
        "우선순위": 5,
        "카테고리": "그래프 확장 (헌재/행정심판 노드)",
        "DB": "Neo4j",
        "이유": "인용 관계 그래프 확장, RAG 컨텍스트 보강",
        "레코드 수": "-",
    },
]

df_priority = pd.DataFrame(priority_data)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_priority.columns),
        fill_color="#4472C4",
        font=dict(color="white", size=12),
        align="left",
    ),
    cells=dict(
        values=[df_priority[col] for col in df_priority.columns],
        fill_color=[["#E2EFDA", "#E2EFDA", "#FFF2CC", "#FFF2CC", "#D9E2F3"]] * len(df_priority.columns),
        align="left",
        font=dict(size=11),
        height=35,
    ),
)])
fig.update_layout(title="DB 구축 우선순위 권고", height=350)
fig.show()

In [11]:
# ── 결과 저장 ─────────────────────────────────────────────
projections = {
    "postgresql": {
        "total_mb": round(total_pg_mb, 1),
        "tables": pg_estimates,
    },
    "lancedb": {
        "total_mb": round(total_lance_mb, 1),
        "total_chunks": int(total_chunks),
        "vector_dim": VECTOR_DIM,
        "categories": lance_estimates,
    },
    "neo4j": {
        "total_mb": round(neo4j_mb, 1),
        "total_nodes": total_nodes,
        "total_edges": total_edges,
        "current": neo4j_current,
        "expanded": neo4j_expanded,
    },
    "summary": {
        "total_files": total_files,
        "total_records": total_records,
        "total_size_gb": round(total_size_gb, 2),
        "total_db_gb": round(total_db_gb, 2),
        "categories": len(CATEGORIES),
    },
}

p7_path = save_result("phase7_projections", projections)
print(f"Phase 7 저장: {p7_path}")

print(f"\n{'='*50}")
print(f"EDA 완료!")
print(f"{'='*50}")
print(f"총 파일: {total_files}개")
print(f"총 레코드: {total_records:,}건")
print(f"원본 크기: {total_size_gb:.1f} GB")
print(f"")
print(f"DB 용량 추정:")
print(f"  PostgreSQL: {total_pg_mb:.0f} MB ({total_pg_mb/1024:.2f} GB)")
print(f"  LanceDB:    {total_lance_mb:.0f} MB ({total_lance_mb/1024:.2f} GB, {total_chunks:,} 청크)")
print(f"  Neo4j:      {neo4j_mb:.0f} MB ({neo4j_mb/1024:.2f} GB, {total_nodes:,} 노드, {total_edges:,} 엣지)")
print(f"  합계:       {(total_pg_mb + total_lance_mb + neo4j_mb):.0f} MB ({total_db_gb:.2f} GB)")

Phase 7 저장: C:\Users\fkjy1\dev\boot_camp\law-3\backend\eda_output\phase7_projections.json

EDA 완료!
총 파일: 48개
총 레코드: 505,412건
원본 크기: 4.8 GB

DB 용량 추정:
  PostgreSQL: 4454 MB (4.35 GB)
  LanceDB:    3705 MB (3.62 GB, 845,212 청크)
  Neo4j:      225 MB (0.22 GB, 136,720 노드, 186,817 엣지)
  합계:       8383 MB (8.19 GB)
